# SecureBERT 2.0 + ASL & Bi-Encoder — Kaggle Training Pipeline

Notebook duy nhất để chạy toàn bộ nghiên cứu trên **Kaggle GPU**:

1. SecureBERT 2.0 + ASL — seed 42 và 123.
2. Bi-Encoder Dense Retrieval — seed 42 và 123.
3. Validation-only threshold tuning, test evaluation, raw predictions.
4. Tổng hợp hai seed, tables, error analysis và figures PNG/PDF từ kết quả đã lưu.

**Cách dùng:** tạo Kaggle Notebook, bật GPU, Add Data chứa thư mục `dataset/processed`, upload notebook này và chọn **Run All**. Kết quả nằm tại `/kaggle/working/results`.

In [ ]:
# Kaggle bootstrap. P100 (Pascal/sm_60) needs a PyTorch wheel that still contains its CUDA kernels.
import subprocess, sys
gpu_name=subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],text=True).strip()
print("Detected GPU:",gpu_name)
if "P100" in gpu_name:
    print("[SETUP] Replacing only torch with the P100-compatible CUDA 12.1 wheel (no dependency upgrades)...")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps","torch==2.5.1","--index-url","https://download.pytorch.org/whl/cu121"])
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers>=4.48,<5","iterative-stratification","sentencepiece","tqdm"])
print("[OK] Kaggle dependencies installed")

In [ ]:
from pathlib import Path
import os, re, gc, json, math, time, random, pickle, platform, subprocess, warnings
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi":120,"axes.grid":True,"grid.alpha":0.25,"font.size":10})

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()
RESULTS = WORK_ROOT / "results"
for d in [RESULTS, RESULTS/"tables", RESULTS/"figures", RESULTS/"figures"/"training",
          RESULTS/"figure_data", RESULTS/"aggregated", RESULTS/"error_analysis"]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seeds": [42],
    "securebert_checkpoint": "cisco-ai/SecureBERT2.0-base",
    "biencoder_checkpoint": "cisco-ai/SecureBERT2.0-biencoder",
    "max_length": 384,
    "max_query_length": 384,
    "max_label_length": 64,
    "validation_fraction": 0.10,
    "securebert_epochs": 6,
    "biencoder_epochs": 4,
    "train_batch_size": 8,
    "eval_batch_size": 16,
    "biencoder_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "biencoder_learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "max_grad_norm": 1.0,
    "gamma_neg": 4.0,
    "gamma_pos": 1.0,
    "asl_clip": 0.05,
    "asl_eps": 1e-8,
    "temperature": 0.07,
    "threshold_min": 0.05,
    "threshold_max": 0.95,
    "threshold_step": 0.01,
    "min_val_support_per_label": 5,
    "top_n_cooccurrence": 20,
    "num_workers": 2,
    "log_every_batches": 250,
    "force_rerun": False,
}
RUN_ALL = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Kaggle GPU is not enabled. Open Notebook settings -> Accelerator -> GPU, then Run All.")
gpu_capability=torch.cuda.get_device_capability(0); compiled_arches=torch.cuda.get_arch_list()
required_arch=f"sm_{gpu_capability[0]}{gpu_capability[1]}"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU capability {gpu_capability} | compiled arches {compiled_arches}")
if compiled_arches and required_arch not in compiled_arches:
    raise RuntimeError(f"Installed PyTorch lacks {required_arch}. Restart the Kaggle session and Run All so the compatibility setup cell runs before torch is imported.")
_cuda_smoke=torch.arange(8,device=DEVICE); torch.cuda.synchronize(); del _cuda_smoke
print("[OK] CUDA kernel smoke test passed")
print("Device:", DEVICE, torch.cuda.get_device_name(0))
print("Results:", RESULTS)
print(json.dumps(CONFIG, indent=2))

## 1. Dataset discovery, integrity checks and validation split

The official test set is never used for checkpoint selection or threshold tuning. A validation subset is derived only from the official training set with iterative multi-label stratification, independently for each seed.

In [ ]:
def find_one(filename, required=True):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches and not IS_KAGGLE:
        matches = sorted(Path.cwd().rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(f"Cannot find {filename}. Add the processed dataset as Kaggle input.")
        return None
    preferred = [p for p in matches if "processed" in str(p).lower()]
    chosen = preferred[0] if preferred else matches[0]
    print(f"[DATA] {filename}: {chosen}")
    return chosen

ORIGINAL_TRAIN_PATH = find_one("train_original_fixed.csv")
import glob
aug_files = glob.glob(str(INPUT_ROOT / "**/train_augmented_*.csv"), recursive=True) + glob.glob(str(WORK_ROOT / "**/train_augmented_*.csv"), recursive=True) + glob.glob(str(Path.cwd() / "**/train_augmented_*.csv"), recursive=True)
if not aug_files: raise FileNotFoundError("Cannot find train_augmented_*.csv")
AUGMENTED_TRAIN_PATH = Path(aug_files[0])

if AUGMENTED_TRAIN_PATH.name == "train_safe_augmented_eda.csv":
    temp_df = pd.read_csv(AUGMENTED_TRAIN_PATH, nrows=120000)
    if 115000 < len(temp_df) < 116000:
        raise RuntimeError(f"REJECTED: File {AUGMENTED_TRAIN_PATH.name} appears to be the OLD leakage-contaminated dataset.")

VALIDATION_PATH = find_one("validation_original_fixed.csv")
TEST_PATH = find_one("test.csv")

print("\n" + "="*70)
print("Resolved original support path:", ORIGINAL_TRAIN_PATH)
print("Resolved augmented train path:", AUGMENTED_TRAIN_PATH)
print("Resolved fixed validation path:", VALIDATION_PATH)
print("Resolved official test path:", TEST_PATH)
print("="*70)

original_train_df = pd.read_csv(ORIGINAL_TRAIN_PATH)
train_df = pd.read_csv(AUGMENTED_TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

MLB_PATH = find_one("multilabel_binarizer.pkl")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open(MLB_PATH, "rb") as f: saved_mlb=pickle.load(f)
classes = np.asarray(saved_mlb.classes_, dtype=str)
mlb=MultiLabelBinarizer(classes=classes); mlb.fit([[]])
NUM_LABELS = len(classes)
label_to_idx = {x:i for i,x in enumerate(classes)}

def parse_labels(value):
    return [x.strip() for x in str(value).split(",") if x.strip() and x.strip() != "nan"]
def encode_labels(series):
    return mlb.transform(series.map(parse_labels)).astype(np.float32)

for frame, name in [(original_train_df,"orig_train"), (train_df,"train"), (val_df,"val"), (test_df,"test")]:
    frame["Cleaned_Text"] = frame["Cleaned_Text"].fillna("").astype(str)

y_original_train = encode_labels(original_train_df["Labels"])
y_train = encode_labels(train_df["Labels"])
y_val = encode_labels(val_df["Labels"])
y_test = encode_labels(test_df["Labels"])

original_train_support = y_original_train.sum(axis=0)
augmented_train_support = y_train.sum(axis=0)

# Sanity Checks
print("\n" + "="*70)
print("RUNNING SANITY CHECKS...")
train_source_ids = set(train_df["source_sample_id"])
val_source_ids = set(val_df.get("source_sample_id", []))
orig_source_ids = set(original_train_df["source_sample_id"])
test_source_ids = set(test_df.get("source_sample_id", []))

if len(train_source_ids.intersection(val_source_ids)) > 0:
    raise RuntimeError("Sanity Check Failed: train source IDs ∩ validation source IDs = 0 violated.")
if len(train_source_ids.intersection(test_source_ids)) > 0:
    raise RuntimeError("Sanity Check Failed: train source IDs ∩ test source IDs = 0 violated.")
if len(val_source_ids.intersection(test_source_ids)) > 0:
    raise RuntimeError("Sanity Check Failed: validation source IDs ∩ test source IDs = 0 violated.")

def normalize_for_duplicate_check(text):
    return re.sub(r'\s+', ' ', str(text).lower()).strip()

norm_train = set(train_df["Cleaned_Text"].apply(normalize_for_duplicate_check))
norm_val = set(val_df["Cleaned_Text"].apply(normalize_for_duplicate_check))
norm_test = set(test_df["Cleaned_Text"].apply(normalize_for_duplicate_check))

overlap_train_val = norm_train.intersection(norm_val)
if len(overlap_train_val) > 0:
    print(f"Exact count of Train-Val text overlap: {len(overlap_train_val)}")
    raise RuntimeError("Sanity Check Failed: normalized train text ∩ normalized validation text = 0 violated.")

overlap_train_test = norm_train.intersection(norm_test)
if len(overlap_train_test) > 0:
    print(f"Exact count of Train-Test text overlap: {len(overlap_train_test)}")
    raise RuntimeError("Sanity Check Failed: normalized train text ∩ normalized test text = 0 violated.")

if "is_augmented" in val_df.columns and val_df["is_augmented"].sum() > 0:
    raise RuntimeError("Sanity Check Failed: val_df['is_augmented'].sum() == 0 violated.")
if "is_augmented" in test_df.columns and test_df["is_augmented"].sum() > 0:
    raise RuntimeError("Sanity Check Failed: test_df['is_augmented'].sum() == 0 violated.")

if "is_augmented" in train_df.columns:
    synth_df = train_df[train_df["is_augmented"] == 1]
    synth_ids = set(synth_df["source_sample_id"])
    if not synth_ids.issubset(orig_source_ids):
        raise RuntimeError("Sanity Check Failed: Synthetic source_sample_id not in original_train_df.source_sample_id violated.")
    
    parent_map = {row["source_sample_id"]: set(parse_labels(row["Labels"])) for _, row in original_train_df.iterrows()}
    for _, row in synth_df.iterrows():
        sid = row["source_sample_id"]
        synth_labels = set(parse_labels(row["Labels"]))
        if sid in parent_map and synth_labels != parent_map[sid]:
            raise RuntimeError(f"Sanity Check Failed: Labels mismatch for synthetic sample {sid}.")

if len(test_df) != 4453:
    print(f"[WARNING] Expected Test size 4453, but found {len(test_df)}")

print("[OK] Sanity checks passed. No leakage detected.")
print("="*70)


In [ ]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

class TextDataset(Dataset):
    def __init__(self, texts, labels, ids):
        self.texts = list(texts); self.labels = np.asarray(labels, np.float32); self.ids = np.asarray(ids)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.labels[i], self.ids[i]

def make_loader(texts, labels, ids, tokenizer, max_length, batch_size, shuffle, seed):
    ds = TextDataset(texts, labels, ids)
    def collate(batch):
        text, y, sample_id = zip(*batch)
        tok = tokenizer(list(text), padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok["labels"] = torch.tensor(np.stack(y), dtype=torch.float32)
        tok["sample_ids"] = np.asarray(sample_id)
        return tok
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate,
                      num_workers=CONFIG["num_workers"], pin_memory=True,
                      worker_init_fn=seed_worker, generator=g, persistent_workers=CONFIG["num_workers"]>0)

## 2. Central metrics and validation-only threshold tuning

In [ ]:
METRIC_KEYS = ["micro_precision","micro_recall","micro_f1","macro_precision","macro_recall",
               "macro_f1","weighted_f1","hamming_loss"]

def classification_metrics(y_true, y_pred):
    return {
        "micro_precision": precision_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_recall": recall_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_f1": f1_score(y_true,y_pred,average="micro",zero_division=0),
        "macro_precision": precision_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_recall": recall_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_f1": f1_score(y_true,y_pred,average="macro",zero_division=0),
        "weighted_f1": f1_score(y_true,y_pred,average="weighted",zero_division=0),
        "hamming_loss": hamming_loss(y_true,y_pred),
        "avg_predicted_labels": float(y_pred.sum(1).mean()),
    }

def ranking_metrics(y_true, scores, ks=(1,3,5,10,20,50)):
    order = np.argsort(-scores, axis=1)
    true_count = np.maximum(y_true.sum(1), 1)
    out = {}
    for k in ks:
        kk=min(k,scores.shape[1]); hits=np.take_along_axis(y_true,order[:,:kk],axis=1).sum(1)
        out[f"precision_at_{k}"] = float(np.mean(hits/kk))
        out[f"recall_at_{k}"] = float(np.mean(hits/true_count))
        out[f"hit_at_{k}"] = float(np.mean(hits>0))
    ranks=[]; aps=[]
    for i in range(len(y_true)):
        rel=y_true[i,order[i]].astype(bool); pos=np.flatnonzero(rel)
        ranks.append((pos[0]+1) if len(pos) else scores.shape[1]+1)
        if len(pos): aps.append(np.mean([(j+1)/(p+1) for j,p in enumerate(pos)]))
        else: aps.append(0.0)
    out["mrr"]=float(np.mean(1/np.asarray(ranks))); out["map"]=float(np.mean(aps))
    return out

def threshold_sweep(y_true, probs):
    thresholds=np.round(np.arange(CONFIG["threshold_min"], CONFIG["threshold_max"]+1e-9,
                                  CONFIG["threshold_step"]), 10)
    rows=[]
    for t in thresholds:
        m=classification_metrics(y_true,(probs>=t).astype(np.uint8)); m["threshold"]=float(t); rows.append(m)
    return pd.DataFrame(rows)

def tune_thresholds(y_true, probs):
    sweep=threshold_sweep(y_true,probs)
    best_micro=float(sweep.loc[sweep.micro_f1.idxmax(),"threshold"])
    best_macro=float(sweep.loc[sweep.macro_f1.idxmax(),"threshold"])
    support=y_true.sum(0).astype(int); per=np.full(y_true.shape[1],best_micro,dtype=np.float32); rows=[]
    for j in range(y_true.shape[1]):
        fallback=support[j] < CONFIG["min_val_support_per_label"]
        if not fallback:
            vals=[]
            for t in sweep.threshold:
                pred=(probs[:,j]>=t).astype(np.uint8)
                vals.append(f1_score(y_true[:,j],pred,zero_division=0))
            per[j]=float(sweep.threshold.iloc[int(np.argmax(vals))])
        pred=(probs[:,j]>=per[j]).astype(np.uint8)
        rows.append({"technique_id":classes[j],"validation_support":support[j],"optimal_threshold":float(per[j]),
                     "validation_precision":precision_score(y_true[:,j],pred,zero_division=0),
                     "validation_recall":recall_score(y_true[:,j],pred,zero_division=0),
                     "validation_f1":f1_score(y_true[:,j],pred,zero_division=0),"fallback_used":bool(fallback)})
    return best_micro,best_macro,per,sweep,pd.DataFrame(rows)

def label_metrics(y_true,y_pred,train_support,val_support,thresholds,groups):
    rows=[]
    for j,tid in enumerate(classes):
        rows.append({"Technique_ID":tid,"Train_Support":int(train_support[j]),"Validation_Support":int(val_support[j]),
                     "Test_Support":int(y_true[:,j].sum()),"Precision":precision_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Recall":recall_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "F1":f1_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Optimal_Threshold":float(thresholds[j]),"Frequency_Group":groups[j]})
    return pd.DataFrame(rows)

def frequency_groups(original_train_support):
    groups = []
    for x in original_train_support:
        if x >= 100: groups.append("Head")
        elif 30 <= x < 100: groups.append("Medium")
        else: groups.append("Tail")
    return np.asarray(groups), 30.0, 100.0

def print_experiment_summary(result):
    m=result["test_per_label"]
    names=[("Micro Precision","micro_precision"),("Micro Recall","micro_recall"),("Micro-F1","micro_f1"),
           ("Macro Precision","macro_precision"),("Macro Recall","macro_recall"),("Macro-F1","macro_f1"),
           ("Weighted-F1","weighted_f1"),("Hamming Loss","hamming_loss"),
           ("Precision@3","precision_at_3"),("Recall@3","recall_at_3"),("Hit@3","hit_at_3"),
           ("Precision@5","precision_at_5"),("Recall@5","recall_at_5"),("Hit@5","hit_at_5")]
    rows=[{"Metric":label,"Test value":m[key]} for label,key in names if key in m]
    print("\n"+"="*70)
    print(f"EXPERIMENT COMPLETE: {result['model']} | Seed {result['seed']}")
    print("="*70)
    print(f"Best epoch: {result['best_epoch']} | Selection: {result['selection_metric']} = {result['validation_score']:.4f}")
    print(f"Validation-selected global threshold: {result['global_threshold']:.2f}")
    display(pd.DataFrame(rows).set_index("Metric").round(4))
    if result.get("retrieval"):
        retrieval=pd.DataFrame([{"Metric":k.replace("_at_","@").replace("_"," ").title(),"Test value":v} for k,v in result["retrieval"].items()])
        print("Bi-Encoder ranking metrics:"); display(retrieval.set_index("Metric").round(4))
    print(f"Training: {result['training_seconds']/60:.2f} min | Inference: {result['inference_ms_per_sample']:.3f} ms/sample")
    print(f"Peak VRAM: {result['peak_vram_mb']:.1f} MB | Model size: {result['model_size_mb']:.1f} MB | Device: {result['device']}")
    print("="*70)

## 3. Models and losses

ASL consumes raw logits and multi-hot targets. Bi-Encoder uses a multi-positive full-label softmax objective: all ground-truth techniques contribute to the numerator, so a second true label is never treated as a negative. The label encoder is frozen and technique embeddings are precomputed, making training feasible on a Kaggle GPU.

In [ ]:
def masked_mean(last_hidden, attention_mask):
    mask=attention_mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden*mask).sum(1)/mask.sum(1).clamp_min(1e-9)

def load_securebert_encoder(checkpoint):
    model_config=AutoConfig.from_pretrained(checkpoint)
    if hasattr(model_config,"reference_compile"): model_config.reference_compile=False
    return AutoModel.from_pretrained(checkpoint,config=model_config,attn_implementation="eager")

class SecureClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels):
        super().__init__(); self.encoder=load_securebert_encoder(checkpoint)
        hidden=self.encoder.config.hidden_size
        self.dropout=nn.Dropout(getattr(self.encoder.config,"classifier_dropout",0.1) or 0.1)
        self.classifier=nn.Linear(hidden,num_labels)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        return self.classifier(self.dropout(masked_mean(out.last_hidden_state,attention_mask)))

class AsymmetricLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,logits,targets):
        p=torch.sigmoid(logits); pos=p; neg=1-p
        if CONFIG["asl_clip"]:
            neg=(neg+CONFIG["asl_clip"]).clamp(max=1)
        loss=targets*torch.log(pos.clamp_min(CONFIG["asl_eps"]))+(1-targets)*torch.log(neg.clamp_min(CONFIG["asl_eps"]))
        pt=pos*targets+neg*(1-targets)
        weight=torch.pow((1-pt).clamp_min(0), CONFIG["gamma_pos"]*targets+CONFIG["gamma_neg"]*(1-targets))
        return -(loss*weight).sum(1).mean()

class Encoder(nn.Module):
    def __init__(self,checkpoint):
        super().__init__(); self.backbone=load_securebert_encoder(checkpoint)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask)
        return F.normalize(masked_mean(out.last_hidden_state,attention_mask),p=2,dim=1)

# ASL sanity checks
_loss=AsymmetricLoss(); _z=torch.tensor([[5.,-5.],[-5.,5.]],requires_grad=True); _y=torch.tensor([[1.,0.],[0.,1.]])
_good=_loss(_z,_y); _bad=_loss(-_z,_y); _good.backward()
assert torch.isfinite(_good) and _good<_bad and _z.grad is not None and torch.isfinite(_z.grad).all()
print("[OK] ASL finite, direction and gradient checks passed")

In [ ]:
def optimizer_and_scheduler(model, loader_len, epochs, lr):
    no_decay=("bias","LayerNorm.weight","layer_norm.weight")
    params=[{"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],"weight_decay":CONFIG["weight_decay"]},
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],"weight_decay":0.0}]
    opt=torch.optim.AdamW(params,lr=lr)
    steps=math.ceil(loader_len/CONFIG["gradient_accumulation_steps"])*epochs
    sch=get_linear_schedule_with_warmup(opt,int(steps*CONFIG["warmup_ratio"]),steps)
    return opt,sch

def to_device(batch):
    ids=batch.pop("sample_ids"); y=batch.pop("labels").to(DEVICE,non_blocking=True)
    x={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
    return x,y,ids

@torch.no_grad()
def predict_classifier(model,loader,desc="Evaluating SecureBERT"):
    model.eval(); ys=[]; probs=[]; ids=[]
    progress=tqdm(loader,desc=desc,leave=False,dynamic_ncols=True)
    for batch in progress:
        x,y,sid=to_device(batch); logits=model(**x)
        ys.append(y.cpu().numpy()); probs.append(torch.sigmoid(logits).cpu().numpy()); ids.extend(sid.tolist())
    return np.concatenate(ys),np.concatenate(probs),np.asarray(ids)

def train_classifier_epoch(model,loader,loss_fn,opt,sch,scaler,desc):
    model.train(); opt.zero_grad(set_to_none=True); total=0
    progress=tqdm(enumerate(loader,1),total=len(loader),desc=desc,leave=True,dynamic_ncols=True)
    for step,batch in progress:
        x,y,_=to_device(batch)
        with torch.autocast("cuda",dtype=torch.float16):
            loss=loss_fn(model(**x),y)/CONFIG["gradient_accumulation_steps"]
        if not torch.isfinite(loss): raise FloatingPointError("NaN/Inf ASL loss")
        scaler.scale(loss).backward(); batch_loss=loss.item()*CONFIG["gradient_accumulation_steps"]; total+=batch_loss
        if step%CONFIG["gradient_accumulation_steps"]==0 or step==len(loader):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG["max_grad_norm"])
            old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale()>=old_scale: sch.step()
            else: print(f"[AMP] Optimizer step skipped after gradient overflow at batch {step}; scheduler unchanged.",flush=True)
            opt.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{batch_loss:.4f}",avg=f"{total/step:.4f}",lr=f"{opt.param_groups[0]['lr']:.2e}",vram=f"{torch.cuda.memory_allocated()/1024**3:.1f}GB")
        if step%CONFIG["log_every_batches"]==0 or step==len(loader): print(f"[PROGRESS] {desc} | batch {step}/{len(loader)} | loss={batch_loss:.4f} | avg={total/step:.4f} | lr={opt.param_groups[0]['lr']:.2e} | VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB",flush=True)
    return total/len(loader)

## 4. SecureBERT 2.0 + ASL training — seed 42 and 123

In [ ]:
def run_securebert(seed):
    run_dir = RESULTS / "securebert_asl_augmented" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir/"metrics.json").exists() and not CONFIG["force_rerun"]:
        print(f"[SKIP] SecureBERT seed {seed} completed"); return
    
    print("\n" + "="*70)
    print(f"TRAINING SECUREBERT 2.0 + ASL | SEED {seed}")
    print("="*70)
    
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); started=time.time()
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["securebert_checkpoint"])
    
    train_loader = make_loader(train_df.Cleaned_Text, y_train, np.arange(len(train_df)), tokenizer, CONFIG["max_length"], CONFIG["train_batch_size"], True, seed)
    val_loader = make_loader(val_df.Cleaned_Text, y_val, np.arange(len(val_df)), tokenizer, CONFIG["max_length"], CONFIG["eval_batch_size"], False, seed)
    test_loader = make_loader(test_df.Cleaned_Text, y_test, np.arange(len(test_df)), tokenizer, CONFIG["max_length"], CONFIG["eval_batch_size"], False, seed)
    
    model = SecureClassifier(CONFIG["securebert_checkpoint"], NUM_LABELS).to(DEVICE)
    opt, sch = optimizer_and_scheduler(model, len(train_loader), CONFIG["securebert_epochs"], CONFIG["learning_rate"])
    scaler = torch.cuda.amp.GradScaler(); loss_fn = AsymmetricLoss()
    
    history = []; best_macro_f1 = -1.0; best_epoch = 0; patience_counter = 0
    checkpoint = run_dir / "best_model.pt"
    
    with open(run_dir / "training.log", "w") as f_log:
        f_log.write("=== SecureBERT Training Log ===\n")
        
        for epoch in range(1, CONFIG["securebert_epochs"]+1):
            t = time.time()
            loss = train_classifier_epoch(model, train_loader, loss_fn, opt, sch, scaler, desc=f"SecureBERT seed {seed} | epoch {epoch}/{CONFIG['securebert_epochs']}")
            epoch_seconds = time.time() - t
            
            vy, vp, _ = predict_classifier(model, val_loader, desc=f"Validation seed {seed} | epoch {epoch}")
            vm = classification_metrics(vy, (vp>=0.5).astype(np.uint8))
            current_val_macro_f1 = vm['macro_f1']
            
            lr = opt.param_groups[0]['lr']
            amp_skipped = 0; amp_rate = 0.0
            
            history.append({
                "epoch": epoch,
                "train_loss": loss,
                "val_micro_precision": vm['micro_precision'],
                "val_micro_recall": vm['micro_recall'],
                "val_micro_f1": vm['micro_f1'],
                "val_macro_precision": vm['macro_precision'],
                "val_macro_recall": vm['macro_recall'],
                "val_macro_f1": vm['macro_f1'],
                "val_weighted_f1": vm['weighted_f1'],
                "val_hamming_loss": vm['hamming_loss'],
                "learning_rate": lr,
                "epoch_seconds": epoch_seconds,
                "amp_skipped_steps": amp_skipped,
                "amp_skip_rate": amp_rate
            })
            
            if current_val_macro_f1 > best_macro_f1:
                best_macro_f1 = current_val_macro_f1
                best_epoch = epoch
                patience_counter = 0
                torch.save(model.state_dict(), checkpoint)
            else:
                patience_counter += 1
                
            log_str = (
                "\n" + "="*70 + "\n" +
                f"SECUREBERT 2.0 + ASL — EPOCH {epoch} / {CONFIG['securebert_epochs']}\n" +
                "="*34 + "\n\n" +
                f"Train Loss:\n{loss:.6f}\n\n" +
                f"Validation Micro Precision:\n{vm['micro_precision']*100:.2f}%\n\n" +
                f"Validation Micro Recall:\n{vm['micro_recall']*100:.2f}%\n\n" +
                f"Validation Micro F1:\n{vm['micro_f1']*100:.2f}%\n\n" +
                f"Validation Macro Precision:\n{vm['macro_precision']*100:.2f}%\n\n" +
                f"Validation Macro Recall:\n{vm['macro_recall']*100:.2f}%\n\n" +
                f"Validation Macro F1:\n{vm['macro_f1']*100:.2f}%\n\n" +
                f"Validation Weighted F1:\n{vm['weighted_f1']*100:.2f}%\n\n" +
                f"Validation Hamming Loss:\n{vm['hamming_loss']:.6f}\n\n" +
                f"Learning Rate:\n{lr:.4e}\n\n" +
                f"Epoch Time:\n{epoch_seconds:.2f} sec\n\n" +
                f"AMP Skipped Steps:\n{amp_skipped}\n\n" +
                f"AMP Skip Rate:\n{amp_rate:.2f}%\n\n" +
                f"Best Epoch So Far:\n{best_epoch}\n\n" +
                f"Best Validation Macro F1 @0.5:\n{best_macro_f1*100:.2f}%\n\n" +
                f"Patience:\n{patience_counter} / 2\n\n" +
                "="*70 + "\n"
            )
            print(log_str)
            f_log.write(log_str)
            
            if patience_counter >= 2:
                early_stop_str = (
                    "\n[EARLY STOPPING]\n" +
                    f"Best Epoch: {best_epoch}\n" +
                    f"Best Validation Macro-F1: {best_macro_f1*100:.2f}%\n"
                )
                print(early_stop_str)
                f_log.write(early_stop_str)
                break
                
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    vy, vp, _ = predict_classifier(model, val_loader, desc=f"Final validation seed {seed}")
    
    global_t, macro_t, per_t, sweep, threshold_table = tune_thresholds(vy, vp)
    sweep.to_csv(run_dir / "threshold_sweep.csv", index=False)
    
    threshold_table_cols = {
        "Technique_ID": classes,
        "Original_Train_Support": original_train_support,
        "Validation_Support": threshold_table["validation_support"],
        "Optimal_Threshold": threshold_table["optimal_threshold"],
        "Validation_Precision": threshold_table["validation_precision"],
        "Validation_Recall": threshold_table["validation_recall"],
        "Validation_F1": threshold_table["validation_f1"],
        "Fallback_Used": threshold_table["fallback_used"]
    }
    pd.DataFrame(threshold_table_cols).to_csv(run_dir / "per_label_thresholds.csv", index=False)
    
    infer_start = time.perf_counter()
    ty, tp, tids = predict_classifier(model, test_loader, desc=f"Locked test inference seed {seed}")
    torch.cuda.synchronize()
    infer_seconds = time.perf_counter() - infer_start
    
    pred_global = (tp >= global_t).astype(np.uint8)
    pred_per = (tp >= per_t[None, :]).astype(np.uint8)
    
    tp_safe = np.clip(tp, 1e-9, 1 - 1e-9)
    tlogits = np.log(tp_safe / (1 - tp_safe))
    
    global_metrics = classification_metrics(ty, pred_global)
    global_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))
    
    per_metrics = classification_metrics(ty, pred_per)
    per_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))
    
    val_support = vy.sum(0).astype(int)
    groups, q25, q75 = frequency_groups(original_train_support)
    
    pl_rows = []
    for j, tid in enumerate(classes):
        fallback = val_support[j] < CONFIG["min_val_support_per_label"]
        pl_rows.append({
            "Technique_ID": tid,
            "Original_Train_Support": int(original_train_support[j]),
            "Augmented_Train_Support": int(augmented_train_support[j]),
            "Validation_Support": int(val_support[j]),
            "Test_Support": int(ty[:,j].sum()),
            "TP": int(((ty[:,j]==1) & (pred_per[:,j]==1)).sum()),
            "FP": int(((ty[:,j]==0) & (pred_per[:,j]==1)).sum()),
            "FN": int(((ty[:,j]==1) & (pred_per[:,j]==0)).sum()),
            "Precision": precision_score(ty[:,j], pred_per[:,j], zero_division=0),
            "Recall": recall_score(ty[:,j], pred_per[:,j], zero_division=0),
            "F1": f1_score(ty[:,j], pred_per[:,j], zero_division=0),
            "Threshold": float(per_t[j]),
            "Threshold_Fallback": bool(fallback),
            "Frequency_Group": groups[j]
        })
    per_label = pd.DataFrame(pl_rows)
    per_label.to_csv(run_dir / "per_label_metrics.csv", index=False)
    
    pd.DataFrame(history).to_csv(run_dir / "epoch_history.csv", index=False)
    
    test_source_sample_ids = test_df.get("source_sample_id", pd.Series([None]*len(tids))).to_numpy()
    np.savez_compressed(run_dir / "predictions.npz", 
                        sample_ids=tids, 
                        source_sample_ids=test_source_sample_ids,
                        y_true=ty, 
                        logits=tlogits,
                        probabilities=tp,
                        predictions_global=pred_global, 
                        predictions_per_label=pred_per)
                        
    params = sum(p.numel() for p in model.parameters())
    model_mb = checkpoint.stat().st_size / 1024**2
    
    dataset_sizes = {
        "original_fixed_train_size": len(original_train_df),
        "original_single_label_count": int((y_original_train.sum(1) == 1).sum()),
        "original_multi_label_count": int((y_original_train.sum(1) > 1).sum()),
        "original_avg_labels_per_sample": float(y_original_train.sum(1).mean()),
        
        "augmented_train_size": len(train_df),
        "synthetic_count": int(len(train_df[train_df.get('is_augmented', 0) == 1])),
        "augmented_avg_labels_per_sample": float(y_train.sum(1).mean()),
        
        "validation_size": len(val_df),
        "validation_avg_labels_per_sample": float(vy.sum(1).mean()),
        
        "test_size": len(test_df),
        "test_avg_labels_per_sample": float(ty.sum(1).mean()),
        
        "num_labels": NUM_LABELS,
        "support_min": int(original_train_support.min()),
        "support_max": int(original_train_support.max()),
        "support_mean": float(original_train_support.mean()),
        "head_count": int((groups == "Head").sum()),
        "medium_count": int((groups == "Medium").sum()),
        "tail_count": int((groups == "Tail").sum()),
        
        "validation_augmented": False,
        "test_augmented": False
    }
    pd.DataFrame([dataset_sizes]).to_csv(run_dir / "dataset_statistics.csv", index=False)
    
    comp_cost = {
        "training_seconds": time.time() - started,
        "inference_ms_per_sample": infer_seconds / len(ty) * 1000,
        "seconds_per_1000_samples": (infer_seconds / len(ty)) * 1000,
        "samples_per_second": len(ty) / infer_seconds,
        "peak_cpu_ram_mb": psutil.Process().memory_info().rss / 1024**2,
        "peak_vram_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "peak_vram_reserved_mb": torch.cuda.max_memory_reserved() / 1024**2,
        "model_size_mb": model_mb,
        "total_parameters": params,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "device": torch.cuda.get_device_name(0)
    }
    pd.DataFrame([comp_cost]).to_csv(run_dir / "computational_cost.csv", index=False)
    
    fg_rows = []
    for gname in ["Head", "Medium", "Tail"]:
        g_df = per_label[per_label["Frequency_Group"] == gname]
        g_df_active = g_df[g_df["Test_Support"] > 0]
        f1_vals = g_df["F1"]
        fg_rows.append({
            "Frequency_Group": gname,
            "n_labels": len(g_df),
            "mean_precision": g_df["Precision"].mean() if len(g_df) else 0,
            "median_precision": g_df["Precision"].median() if len(g_df) else 0,
            "mean_recall": g_df["Recall"].mean() if len(g_df) else 0,
            "median_recall": g_df["Recall"].median() if len(g_df) else 0,
            "mean_f1": f1_vals.mean() if len(g_df) else 0,
            "median_f1": f1_vals.median() if len(g_df) else 0,
            "f1_zero_count": int((f1_vals == 0).sum()),
            "f1_zero_percentage": float((f1_vals == 0).mean() * 100) if len(g_df) else 0,
            "f1_one_count": int((f1_vals == 1).sum()),
            "f1_one_percentage": float((f1_vals == 1).mean() * 100) if len(g_df) else 0,
            "total_test_support": int(g_df["Test_Support"].sum())
        })
    fg_df = pd.DataFrame(fg_rows)
    fg_df.to_csv(run_dir / "frequency_group_performance.csv", index=False)
    
    tail_present = per_label[(per_label["Frequency_Group"] == "Tail") & (per_label["Test_Support"] > 0)]
    tp_count = len(tail_present)
    
    result = {
        "experiment_name": f"SecureBERT_seed_{seed}",
        "model": "SecureBERT 2.0 + ASL",
        "loss": "Asymmetric Loss",
        "seed": seed,
        "protocol_version": "leakage-safe",
        "augmentation": True,
        "augmentation_mode": "eda",
        "train_file": AUGMENTED_TRAIN_PATH.name,
        "original_train_file": ORIGINAL_TRAIN_PATH.name,
        "validation_file": VALIDATION_PATH.name,
        "test_file": TEST_PATH.name,
        
        "best_epoch": best_epoch,
        "selection_metric": "Validation Macro-F1 @ 0.5",
        "best_validation_macro_f1_at_05": best_macro_f1,
        
        "validation_metrics": vm,
        "micro_optimal_validation_threshold": float(global_t),
        "macro_optimal_validation_threshold": float(macro_t),
        "selected_global_threshold": float(global_t),
        
        "test_global": global_metrics,
        "test_per_label": per_metrics,
        
        "ranking_metrics": ranking_metrics(ty, tp, ks=(3, 5)),
        
        "true_test_cardinality": dataset_sizes["test_avg_labels_per_sample"],
        "predicted_test_cardinality_global": float(pred_global.sum(1).mean()),
        "predicted_test_cardinality_per_label": float(pred_per.sum(1).mean()),
        
        "frequency_group_metrics": fg_df.to_dict("records"),
        "tail_present_metrics": {
            "number_of_labels": tp_count,
            "mean_f1": float(tail_present["F1"].mean()) if tp_count else 0,
            "median_f1": float(tail_present["F1"].median()) if tp_count else 0,
            "f1_zero_count": int((tail_present["F1"] == 0).sum()),
            "f1_zero_percentage": float((tail_present["F1"] == 0).mean() * 100) if tp_count else 0,
        },
        
        **comp_cost,
        **dataset_sizes
    }
    (run_dir / "metrics.json").write_text(json.dumps(result, indent=2))
    tokenizer.save_pretrained(run_dir / "tokenizer")
    
    final_output = (
        "\n" + "="*70 + "\n" +
        "FINAL TEST — GLOBAL THRESHOLD\n" +
        "="*29 + "\n\n" +
        f"Micro Precision\n{global_metrics['micro_precision']*100:.2f}%\n\n" +
        f"Micro Recall\n{global_metrics['micro_recall']*100:.2f}%\n\n" +
        f"Micro F1\n{global_metrics['micro_f1']*100:.2f}%\n\n" +
        f"Macro Precision\n{global_metrics['macro_precision']*100:.2f}%\n\n" +
        f"Macro Recall\n{global_metrics['macro_recall']*100:.2f}%\n\n" +
        f"Macro F1\n{global_metrics['macro_f1']*100:.2f}%\n\n" +
        f"Weighted F1\n{global_metrics['weighted_f1']*100:.2f}%\n\n" +
        f"Hamming Loss\n{global_metrics['hamming_loss']:.6f}\n\n" +
        f"P@3\n{global_metrics.get('precision_at_3', 0)*100:.2f}%\n\n" +
        f"R@3\n{global_metrics.get('recall_at_3', 0)*100:.2f}%\n\n" +
        f"Hit@3\n{global_metrics.get('hit_at_3', 0)*100:.2f}%\n\n" +
        f"P@5\n{global_metrics.get('precision_at_5', 0)*100:.2f}%\n\n" +
        f"R@5\n{global_metrics.get('recall_at_5', 0)*100:.2f}%\n\n" +
        f"Hit@5\n{global_metrics.get('hit_at_5', 0)*100:.2f}%\n\n" +
        f"MRR\n{global_metrics.get('mrr', 0):.4f}\n\n" +
        f"MAP\n{global_metrics.get('map', 0):.4f}\n\n" +
        f"True Labels / Sample\n{dataset_sizes['test_avg_labels_per_sample']:.4f}\n\n" +
        f"Predicted Labels / Sample\n{float(pred_global.sum(1).mean()):.4f}\n\n" +
        f"Global Threshold\n{global_t:.2f}\n\n" +
        "="*70 + "\n\n" +
        
        "="*70 + "\n" +
        "FINAL TEST — PER-LABEL THRESHOLDS\n" +
        "="*33 + "\n\n" +
        f"Micro Precision\n{per_metrics['micro_precision']*100:.2f}%\n\n" +
        f"Micro Recall\n{per_metrics['micro_recall']*100:.2f}%\n\n" +
        f"Micro F1\n{per_metrics['micro_f1']*100:.2f}%\n\n" +
        f"Macro Precision\n{per_metrics['macro_precision']*100:.2f}%\n\n" +
        f"Macro Recall\n{per_metrics['macro_recall']*100:.2f}%\n\n" +
        f"Macro F1\n{per_metrics['macro_f1']*100:.2f}%\n\n" +
        f"Weighted F1\n{per_metrics['weighted_f1']*100:.2f}%\n\n" +
        f"Hamming Loss\n{per_metrics['hamming_loss']:.6f}\n\n" +
        f"Predicted Labels / Sample\n{float(pred_per.sum(1).mean()):.4f}\n\n" +
        "="*70 + "\n"
    )
    print(final_output)
    
    del model, opt, sch, scaler, train_loader, val_loader, test_loader; gc.collect(); torch.cuda.empty_cache()


## 5. Bi-Encoder Dense Retrieval training — seed 42 and 123

No external ATT&CK metadata file is required. Technique representations use only the 378 Technique IDs from `multilabel_binarizer.pkl`. This is recorded as an **ID-only Bi-Encoder** configuration in every run.

In [ ]:
# [DELETED] Bi-Encoder logic removed as requested.


## 6. Aggregate actual outputs and generate paper figures

This section only reads saved JSON/CSV/NPZ files. It never starts training and never replaces missing values with zero.

In [ ]:
def savefig(fig,name):
    fig.tight_layout(); fig.savefig(RESULTS/"figures"/f"{name}.png",dpi=300,bbox_inches="tight")
    fig.savefig(RESULTS/"figures"/f"{name}.pdf",bbox_inches="tight")
    fig.savefig(RESULTS/"figures"/f"{name}.svg",bbox_inches="tight")
    plt.close(fig)

def read_metric(seed):
    p = RESULTS / "securebert_asl_augmented" / f"seed_{seed}" / "metrics.json"
    return json.loads(p.read_text()) if p.exists() else None

def generate_securebert_figures(seed):
    run_dir = RESULTS / "securebert_asl_augmented" / f"seed_{seed}"
    if not (run_dir / "metrics.json").exists():
        print(f"[ERROR] Cannot generate figures, metrics.json missing for seed {seed}")
        return
        
    groups, _, _ = frequency_groups(original_train_support)
    d = pd.DataFrame({"Technique_ID": classes, "Train_Support": original_train_support, "Frequency_Group": groups}).sort_values("Train_Support", ascending=False)
    d.to_csv(run_dir / "figure_data" / "fig1_original_label_support.csv", index=False)
    fig, ax = plt.subplots(figsize=(10, max(12, NUM_LABELS * .16)))
    ax.barh(d.Technique_ID[::-1], d.Train_Support[::-1], color="#376795")
    ax.set(xlabel="Number of original training samples", ylabel="Technique ID", title="Original Training Label Distribution (Head >=100, Medium 30-99, Tail <30)")
    ax.tick_params(axis="y", labelsize=5)
    savefig(fig, "fig1_original_label_support")

    counts = original_train_df["Labels"].map(lambda x: len(parse_labels(x)))
    cats = pd.Categorical(counts.map(lambda x: str(x) if x < 4 else ">=4"), categories=["1", "2", "3", ">=4"], ordered=True)
    d2 = pd.Series(cats).value_counts(sort=False).rename_axis("Label_Count_Group").reset_index(name="Sample_Count")
    d2["Percentage"] = 100 * d2.Sample_Count / d2.Sample_Count.sum()
    d2.to_csv(run_dir / "figure_data" / "fig2_labels_per_sample.csv", index=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(d2.Label_Count_Group, d2.Sample_Count, color="#4C956C")
    for b, (_, r) in zip(bars, d2.iterrows()):
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{int(r.Sample_Count):,}\n({r.Percentage:.1f}%)", ha="center", va="bottom")
    ax.set(xlabel="Labels per sample", ylabel="Sample count", title="Number of labels per CTI sample (Original)")
    savefig(fig, "fig2_labels_per_sample")

    top = np.argsort(-original_train_support)[:CONFIG["top_n_cooccurrence"]]
    co = y_original_train[:, top].T @ y_original_train[:, top]
    co_df = pd.DataFrame(co.astype(int), index=classes[top], columns=classes[top])
    co_df.to_csv(run_dir / "figure_data" / "fig3_label_cooccurrence.csv")
    fig, ax = plt.subplots(figsize=(11, 9))
    image = ax.imshow(co_df.to_numpy(), cmap="Blues", aspect="auto")
    ax.grid(False)
    ax.set_xticks(range(len(co_df)), co_df.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(co_df)), co_df.index)
    fig.colorbar(image, ax=ax, label="Co-occurrence count")
    ax.set_title("Label co-occurrence (top 20 original training labels)")
    savefig(fig, "fig3_label_cooccurrence")

    h_path = run_dir / "epoch_history.csv"
    if h_path.exists():
        h = pd.read_csv(h_path)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(h.epoch, h.train_loss, marker="o", color="#d62828")
        ax.set(xlabel="Epoch", ylabel="Train Loss", title="Training Loss by Epoch")
        savefig(fig, "fig4_training_loss")
        
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(h.epoch, h.val_micro_f1, marker="o", label="Micro-F1")
        ax.plot(h.epoch, h.val_macro_f1, marker="s", label="Macro-F1")
        ax.set(xlabel="Epoch", ylabel="Score", title="Validation F1 by Epoch")
        ax.legend()
        savefig(fig, "fig5_validation_f1")

    sweep_path = run_dir / "threshold_sweep.csv"
    if sweep_path.exists():
        sweep = pd.read_csv(sweep_path)
        fig, ax = plt.subplots(figsize=(8, 5))
        for col, label in [("micro_f1", "Micro-F1"), ("macro_f1", "Macro-F1"), ("micro_precision", "Micro Precision"), ("micro_recall", "Micro Recall")]:
            ax.plot(sweep.threshold, sweep[col], label=label)
        best_t = sweep.loc[sweep.micro_f1.idxmax(), "threshold"]
        ax.axvline(best_t, ls="--", color="black", label=f"Best Micro-F1 ({best_t})")
        ax.set(xlabel="Global validation threshold", ylabel="Score", title="Validation Threshold Sweep")
        ax.legend()
        savefig(fig, "fig6_threshold_sweep")
        
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(sweep.threshold, sweep.avg_predicted_labels, color="#9C6644")
        true_card = float(y_val.sum(1).mean())
        ax.axhline(true_card, ls="--", color="gray", label=f"True Cardinality ({true_card:.2f})")
        ax.set(xlabel="Global validation threshold", ylabel="Average predicted labels/sample", title="Threshold vs Predicted Cardinality")
        ax.legend()
        savefig(fig, "fig7_threshold_cardinality")

    pl_path = run_dir / "per_label_metrics.csv"
    if pl_path.exists():
        pl = pd.read_csv(pl_path).sort_values("F1", ascending=False)
        fig, ax = plt.subplots(figsize=(10, max(12, NUM_LABELS * .16)))
        ax.barh(pl.Technique_ID[::-1], pl.F1[::-1], color="#6A4C93")
        ax.tick_params(axis="y", labelsize=5)
        ax.set(xlabel="Test F1", ylabel="Technique ID", title="SecureBERT 2.0 + ASL Per-Technique Test F1")
        savefig(fig, "fig8_per_technique_f1")
        
        fg_path = run_dir / "frequency_group_performance.csv"
        if fg_path.exists():
            fg = pd.read_csv(fg_path).set_index("Frequency_Group")
            cols = ["mean_precision", "mean_recall", "mean_f1"]
            if set(cols).issubset(fg.columns):
                fg_plot = fg[cols].reindex(["Head", "Medium", "Tail"])
                fig, ax = plt.subplots(figsize=(8, 5))
                fg_plot.plot.bar(ax=ax)
                ax.set(xlabel="Frequency Group (Original Support)", ylabel="Mean per-label score", title="Performance by Label Frequency")
                ax.tick_params(axis="x", rotation=0)
                savefig(fig, "fig9_frequency_group_scores")
                
        fig, ax = plt.subplots(figsize=(8, 5))
        colors = {"Head": "#277DA1", "Medium": "#F9C74F", "Tail": "#F94144"}
        for group_name, group_df in pl.groupby("Frequency_Group"):
            ax.scatter(group_df.Original_Train_Support, group_df.F1, label=group_name, color=colors.get(group_name), alpha=.75)
        ax.set_xscale("log")
        ax.legend(title="Frequency group")
        ax.set(xlabel="Original Training label support (log scale)", ylabel="Test F1", title="Original Label Support vs Test F1")
        savefig(fig, "fig10_support_vs_f1")

    m = read_metric(seed)
    if m:
        rows = [
            {"Configuration": "Global threshold", "Metric": "Micro-F1", "Score": m["test_global"]["micro_f1"]},
            {"Configuration": "Global threshold", "Metric": "Macro-F1", "Score": m["test_global"]["macro_f1"]},
            {"Configuration": "Global threshold", "Metric": "Weighted-F1", "Score": m["test_global"]["weighted_f1"]},
            {"Configuration": "Per-label threshold", "Metric": "Micro-F1", "Score": m["test_per_label"]["micro_f1"]},
            {"Configuration": "Per-label threshold", "Metric": "Macro-F1", "Score": m["test_per_label"]["macro_f1"]},
            {"Configuration": "Per-label threshold", "Metric": "Weighted-F1", "Score": m["test_per_label"]["weighted_f1"]}
        ]
        df = pd.DataFrame(rows)
        df.to_csv(run_dir / "figure_data" / "fig11_global_vs_perlabel.csv", index=False)
        pivot = df.pivot(index="Metric", columns="Configuration", values="Score")
        fig, ax = plt.subplots(figsize=(8, 5))
        pivot.plot.bar(ax=ax, color=["#577590", "#F9844A"])
        ax.set(ylabel="Test Score", title="Global vs Per-label Threshold Performance")
        ax.tick_params(axis="x", rotation=0)
        savefig(fig, "fig11_global_vs_perlabel")
        
    print(f"[OK] SecureBERT Figures generated in {run_dir/'figures'}")


## 7. Run the complete pipeline

This is the only execution cell needed. Completed runs are skipped unless `CONFIG["force_rerun"] = True`. Figure generation runs only after all four metrics files exist.

In [ ]:
if RUN_ALL:
    for seed in CONFIG["seeds"]:
        run_securebert(seed)
        generate_securebert_figures(seed)
